In [1]:
from  dotenv import load_dotenv

from tools import web_search, agent, response

from memory import connection, checkpointer

load_dotenv()

True

In [2]:
from langchain.chat_models import init_chat_model
import os
model = init_chat_model(
    model = "qwen3.8-flash",
    model_provider="openai",
    base_url = os.getenv("DASHSCOPE_BASE_URL"),
    api_key = os.getenv("DASHSCOPE_API_KEY"),
)

In [3]:
from langchain_tavily import TavilySearch
web_search = TavilySearch(
    max_results=5,
    topic = "general"
)

In [4]:
from langgraph.checkpoint.sqlite import SqliteSaver
import  sqlite3
connection = sqlite3.connect("resources/personal_chief.db", check_same_thread=False)
checkpointer = SqliteSaver(connection)
checkpointer.setup()

In [5]:
from langchain.agents import create_agent
system_prompt = """
你是一名私人厨师。收到用户提供的食材照片或清单后，请按以下流程操作：
1.识别和评估食材：若用户提供照片，首先辨识所有可见食材。基于食材的外观状态，评估其新鲜度与可用量，整理出一份“当前可用食材清单”。
2.智能食谱检索：优先调用 web_search 工具，以“可用食材清单”为核心关键词，查找可行菜谱。
3.多维度评估与排序：从营养价值和制作难度两个维度对检索到的候选食谱进行量化打分，并根据得分排序，制作简单且营养丰富的排名靠前。
4.结构化方案输出：把排序后的食谱整理为一份结构清晰的建议报告，要包含食谱信息、得分、推荐理由、食谱的参考图片，帮助用户快速做出决策。

请严格按照流程，优先调用 web_search 工具搜索食谱，搜索不到的情况下才能自己发挥。


"""
agent = create_agent(
    model = model,
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=checkpointer
)

In [6]:
from langchain.messages import HumanMessage

multimodel_message = HumanMessage([
    {"type":"text","text":"帮我看看能做什么" },
    {"type":"image","url":"https://img95.699pic.com/photo/50764/1552.jpg_wh860.jpg"}
])
config = {"configurable":{"thread_id":"1"}}

In [7]:
response = agent.invoke({"messages":[multimodel_message]},config)

In [8]:
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

[{'type': 'text', 'text': '帮我看看能做什么'}, {'type': 'image', 'url': 'https://img95.699pic.com/photo/50764/1552.jpg_wh860.jpg'}]
================================== Ai Message ==================================

我先仔细辨认了冰箱里的食材，下面整理出清单，然后去检索可行菜谱。

**当前可用食材清单（基于图片识别）**

- 蔬菜类：西兰花/绿叶菜、黄甜椒、黄瓜、西红柿、胡萝卜
- 水果类：橙子、青苹果/柠檬、香蕉、苹果、草莓/浆果
- 蛋奶类：鸡蛋、牛奶、酸奶、奶酪
- 主食/豆制品：面包、盒装豆腐/蛋盒、土豆、洋葱、甜甜圈
- 调味/饮品：果汁、番茄酱、橄榄油/酒、盒装饮料

下面我去检索以这些食材为主的家常菜谱。
Tool Calls:
  tavily_search (call_4347572526c149ce870bd103)
 Call ID: call_4347572526c149ce870bd103
  Args:
    query: 西兰花 胡萝卜 西红柿 鸡蛋 家常菜谱 做法
    search_depth: advanced
  tavily_search (call_8c1714ddb8314a76a3db5217)
 Call ID: call_8c1714ddb8314a76a3db5217
  Args:
    query: 黄瓜 西红柿 鸡蛋 豆腐 快手菜 食谱 做法
    search_depth: advanced
  tavily_search (call_5fd8fc3ee8ff41ce97912d9c)
 Call ID: call_5fd8fc3ee8ff41ce97912d9c
  Args:
    query: 香蕉 牛奶 鸡蛋 健康食谱 做法 早餐
    search_depth: advanced
========================

In [9]:
response = agent.invoke(
    {"messages":[HumanMessage(content="我喜欢第三道菜，可以说的更详细点吗？")]},
    config
)

In [10]:
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

[{'type': 'text', 'text': '帮我看看能做什么'}, {'type': 'image', 'url': 'https://img95.699pic.com/photo/50764/1552.jpg_wh860.jpg'}]
================================== Ai Message ==================================

我先仔细辨认了冰箱里的食材，下面整理出清单，然后去检索可行菜谱。

**当前可用食材清单（基于图片识别）**

- 蔬菜类：西兰花/绿叶菜、黄甜椒、黄瓜、西红柿、胡萝卜
- 水果类：橙子、青苹果/柠檬、香蕉、苹果、草莓/浆果
- 蛋奶类：鸡蛋、牛奶、酸奶、奶酪
- 主食/豆制品：面包、盒装豆腐/蛋盒、土豆、洋葱、甜甜圈
- 调味/饮品：果汁、番茄酱、橄榄油/酒、盒装饮料

下面我去检索以这些食材为主的家常菜谱。
Tool Calls:
  tavily_search (call_4347572526c149ce870bd103)
 Call ID: call_4347572526c149ce870bd103
  Args:
    query: 西兰花 胡萝卜 西红柿 鸡蛋 家常菜谱 做法
    search_depth: advanced
  tavily_search (call_8c1714ddb8314a76a3db5217)
 Call ID: call_8c1714ddb8314a76a3db5217
  Args:
    query: 黄瓜 西红柿 鸡蛋 豆腐 快手菜 食谱 做法
    search_depth: advanced
  tavily_search (call_5fd8fc3ee8ff41ce97912d9c)
 Call ID: call_5fd8fc3ee8ff41ce97912d9c
  Args:
    query: 香蕉 牛奶 鸡蛋 健康食谱 做法 早餐
    search_depth: advanced
========================